# 12. Multiple Alignment, Conservation and Motifs

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `show_alignment.py`

Print part of the globin alignment with a conservation line under it.


In [ ]:
# Print part of the globin alignment with a conservation line under it.
import sys

from Bio import AlignIO

alignment = AlignIO.read("globins_aligned.fasta", "fasta")
start, end = (int(n) for n in sys.argv[1:3])  # columns, counted from 1

for record in alignment:
    name = record.id.split("|")[2]
    print(f"{name:11} {record.seq[start - 1 : end]}")

marks = []
for column in range(start - 1, end):
    letters = set(alignment[:, column])
    beta_letters = set(alignment[:12, column])
    if len(letters) == 1 and "-" not in letters:
        marks.append("*")  # identical in all sixteen
    elif len(beta_letters) == 1 and "-" not in beta_letters:
        marks.append(".")  # identical in the twelve beta globins
    else:
        marks.append(" ")
print(" " * 12 + "".join(marks))

## `conservation.py`

How strongly each column of the globin alignment is conserved, and which columns are identical in all sixteen proteins.


In [ ]:
# How strongly each column of the globin alignment is conserved, and which
# columns are identical in all sixteen proteins.
import math
from collections import Counter

from Bio import AlignIO

alignment = AlignIO.read("globins_aligned.fasta", "fasta")
human = alignment[0].seq  # human beta globin, the reference for numbering


def entropy(column: str) -> float:
    """Shannon entropy in bits: 0 if all letters agree, higher if not."""
    counts = Counter(column)
    total = len(column)
    return -sum(n / total * math.log2(n / total) for n in counts.values())


position = 0  # traditional count: valine after the first methionine is 1
rows = []
for index in range(alignment.get_alignment_length()):
    column = alignment[:, index]
    if human[index] != "-":
        position += 1
    rows.append((index + 1, position - 1, column, entropy(column)))

invariant = [r for r in rows if len(set(r[2])) == 1 and r[2][0] != "-"]
mean = sum(r[3] for r in rows) / len(rows)
print(f"{len(rows)} columns, mean entropy {mean:.2f} bits")
print(f"{len(invariant)} columns identical in all sixteen proteins:")
for column, beta, letters, _ in invariant:
    print(f"  column {column:3}  beta {letters[0]}{beta}")

## `motif.py`

The motif around the proximal histidine, the residue that holds the heme iron: a position weight matrix and a sequence logo.


In [ ]:
# The motif around the proximal histidine, the residue that holds the
# heme iron: a position weight matrix and a sequence logo.
import math

import logomaker
import matplotlib
from Bio import AlignIO, motifs

matplotlib.use("svg")
import matplotlib.pyplot as plt

FIRST, LAST = 89, 100  # alignment columns, counted from 1; no gaps here
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

alignment = AlignIO.read("globins_aligned.fasta", "fasta")
window = [str(r.seq[FIRST - 1 : LAST]) for r in alignment]
motif = motifs.create(window, alphabet=AMINO_ACIDS)
print(f"{len(window)} instances, consensus {motif.consensus}")

frequencies = motif.counts.normalize()
print("column  most common  share  information (bits)")
for i in range(len(motif)):
    shares = {aa: frequencies[aa][i] for aa in AMINO_ACIDS}
    entropy = -sum(p * math.log2(p) for p in shares.values() if p > 0)
    top = max(shares, key=shares.get)
    information = math.log2(len(AMINO_ACIDS)) - entropy
    share = shares[top]
    print(f"{FIRST + i:6}  {top:>11}  {share:5.2f}  {information:8.2f}")

counts = logomaker.alignment_to_matrix(window)
bits = logomaker.transform_matrix(
    counts, from_type="counts", to_type="information", pseudocount=0
)
fig, ax = plt.subplots(figsize=(5.0, 2.2))
logomaker.Logo(bits, ax=ax, color_scheme="black", font_name="DejaVu Sans")
ax.set_xticks(range(len(window[0])))
ax.set_xticklabels(range(FIRST, LAST + 1))
ax.set_ylim(0, math.log2(len(AMINO_ACIDS)))
ax.set_ylabel("bits")
fig.savefig("proximal-his-logo.svg", bbox_inches="tight")
print("logo saved as proximal-his-logo.svg")